# Phase 7: Hybrid Deep Learning Model

This notebook trains two Hybrid Deep Learning models (combining Sentinel-2 image patches and OSM features), validates them using a 2-stage unfreezing schedule (5 epochs frozen, 5 epochs fine-tuning blocks 7 & 8), compares them against classical ML baselines, and generates Grad-CAM explainability heatmaps.

In [ ]:
import os
import json
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models

%matplotlib inline
sns.set_theme(style='whitegrid')

## 1. Grad-CAM Definition
We define a native Grad-CAM class to hook into the last layer of EfficientNet-B0's feature extractor (`backbone.features[-1]`).

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.activations = None
        self.gradients = None
        self.forward_hook = target_layer.register_forward_hook(self.save_activation)
        self.backward_hook = target_layer.register_full_backward_hook(self.save_gradient)
        
    def save_activation(self, module, input, output):
        self.activations = output
        
    def save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0]
        
    def __call__(self, x_img, x_tab, class_idx):
        self.model.zero_grad()
        logits = self.model(x_img, x_tab)
        loss = logits[0, class_idx]
        loss.backward()
        
        acts = self.activations[0].detach().cpu().numpy()
        grads = self.gradients[0].detach().cpu().numpy()
        weights = np.mean(grads, axis=(1, 2))
        
        cam = np.zeros(acts.shape[1:], dtype=np.float32)
        for i, w in enumerate(weights):
            cam += w * acts[i, :, :]
            
        cam = np.maximum(cam, 0)
        cam = cam - np.min(cam)
        cam = cam / (np.max(cam) + 1e-8)
        return cam
        
    def release(self):
        self.forward_hook.remove()
        self.backward_hook.remove()

## 2. Dataset & Model Definitions
We load patches in-memory once to cache Stage 1 (embeddings) and Stage 2 (intermediate features) to speed up CPU training.

In [ ]:
class CachedHybridDataset(Dataset):
    def __init__(self, indices, X_img_cached, X_tab, y):
        self.indices = indices
        self.X_img_cached = X_img_cached
        self.X_tab = torch.tensor(X_tab, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        
    def __len__(self):
        return len(self.indices)
        
    def __getitem__(self, idx):
        global_idx = self.indices[idx]
        x_img = torch.tensor(self.X_img_cached[global_idx], dtype=torch.float32)
        x_tab = self.X_tab[idx]
        label = self.y[idx]
        return x_img, x_tab, label

class HybridModel(nn.Module):
    def __init__(self, tabular_dim):
        super().__init__()
        self.backbone = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        old_conv = self.backbone.features[0][0]
        self.backbone.features[0][0] = nn.Conv2d(
            in_channels=6,
            out_channels=old_conv.out_channels,
            kernel_size=old_conv.kernel_size,
            stride=old_conv.stride,
            padding=old_conv.padding,
            bias=old_conv.bias
        )
        with torch.no_grad():
            self.backbone.features[0][0].weight[:, :3, :, :] = old_conv.weight
            self.backbone.features[0][0].weight[:, 3:, :, :] = old_conv.weight
            
        self.backbone.classifier = nn.Identity()
        
        self.tab_branch = nn.Sequential(
            nn.Linear(tabular_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32)
        )
        
        self.fusion = nn.Sequential(
            nn.Linear(1280 + 32, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 3)
        )
        
    def forward_stage1(self, img_emb, x_tab):
        tab_emb = self.tab_branch(x_tab)
        x = torch.cat([img_emb, tab_emb], dim=1)
        logits = self.fusion(x)
        return logits
        
    def forward_stage2(self, intermediate_feat, x_tab):
        x_img = self.backbone.features[7](intermediate_feat)
        x_img = self.backbone.features[8](x_img)
        x_img = self.backbone.avgpool(x_img)
        img_emb = torch.flatten(x_img, 1)
        
        tab_emb = self.tab_branch(x_tab)
        x = torch.cat([img_emb, tab_emb], dim=1)
        logits = self.fusion(x)
        return logits

    def forward(self, x_img, x_tab):
        img_emb = self.backbone(x_img)
        tab_emb = self.tab_branch(x_tab)
        x = torch.cat([img_emb, tab_emb], dim=1)
        logits = self.fusion(x)
        return logits

## 3. Load Dataset & In-Memory Pre-loading

In [ ]:
project_root = Path('..')
df_cnn = pd.read_csv(project_root / 'data/cnn/cnn_dataset.csv')

print('Pre-loading Sentinel-2 image patches into RAM...')
all_images = np.zeros((len(df_cnn), 6, 128, 128), dtype=np.float32)
for idx, row in df_cnn.iterrows():
    img_rel_path = row['image_path']
    all_images[idx] = np.load(project_root / img_rel_path)
print('Pre-loading completed.')

## 4. Train Model Helper

In [ ]:
def train_model(model_name, tabular_cols, df_train, df_val, train_indices, val_indices, y_train_enc, y_val_enc, device):
    scaler = StandardScaler()
    X_train_tab = scaler.fit_transform(df_train[tabular_cols])
    X_val_tab = scaler.transform(df_val[tabular_cols])
    
    model = HybridModel(tabular_dim=len(tabular_cols)).to(device)
    model.eval()
    
    # Pre-extract stage 1 image embeddings
    img_embs = []
    with torch.no_grad():
        for offset in range(0, len(all_images), 128):
            batch_imgs = torch.tensor(all_images[offset:offset+128], dtype=torch.float32).to(device)
            emb = model.backbone(batch_imgs)
            img_embs.append(emb.cpu().numpy())
    img_embs = np.concatenate(img_embs, axis=0)
    
    # Pre-extract stage 2 intermediate features
    intermediate_feats = []
    with torch.no_grad():
        for offset in range(0, len(all_images), 128):
            batch_imgs = torch.tensor(all_images[offset:offset+128], dtype=torch.float32).to(device)
            x = batch_imgs
            for i in range(7):
                x = model.backbone.features[i](x)
            intermediate_feats.append(x.cpu().numpy())
    intermediate_feats = np.concatenate(intermediate_feats, axis=0)
    
    # Stage 1 Dataset Loader
    train_dataset = CachedHybridDataset(train_indices, img_embs, X_train_tab, y_train_enc)
    val_dataset = CachedHybridDataset(val_indices, img_embs, X_val_tab, y_val_enc)
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
    
    criterion = nn.CrossEntropyLoss()
    for param in model.backbone.parameters():
        param.requires_grad = False
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=1)
    
    best_f1 = 0.0
    best_model_state = None
    best_val_report = None
    best_cm = None
    
    for epoch in range(10):
        if epoch == 5:
            print(f'[{model_name}] Epoch 6: Switching to Stage 2 unfreezing...')
            for param in model.backbone.features[7].parameters():
                param.requires_grad = True
            for param in model.backbone.features[8].parameters():
                param.requires_grad = True
                
            train_dataset = CachedHybridDataset(train_indices, intermediate_feats, X_train_tab, y_train_enc)
            val_dataset = CachedHybridDataset(val_indices, intermediate_feats, X_val_tab, y_val_enc)
            train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
            val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
            
            optimizer = optim.Adam([
                {'params': filter(lambda p: p.requires_grad and not any(id(p) == id(bp) for bp in list(model.backbone.features[7].parameters()) + list(model.backbone.features[8].parameters())), model.parameters()), 'lr': 1e-3},
                {'params': list(model.backbone.features[7].parameters()) + list(model.backbone.features[8].parameters()), 'lr': 1e-5}
            ])
            scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=1)
            
        model.train()
        train_loss = 0.0
        for x_img, x_tab, y in train_loader:
            x_img, x_tab, y = x_img.to(device), x_tab.to(device), y.to(device)
            optimizer.zero_grad()
            if epoch < 5:
                logits = model.forward_stage1(x_img, x_tab)
            else:
                logits = model.forward_stage2(x_img, x_tab)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * x_img.size(0)
        train_loss /= len(train_loader.dataset)
        
        model.eval()
        val_loss = 0.0
        all_preds = []
        all_targets = []
        with torch.no_grad():
            for x_img, x_tab, y in val_loader:
                x_img, x_tab, y = x_img.to(device), x_tab.to(device), y.to(device)
                if epoch < 5:
                    logits = model.forward_stage1(x_img, x_tab)
                else:
                    logits = model.forward_stage2(x_img, x_tab)
                loss = criterion(logits, y)
                val_loss += loss.item() * x_img.size(0)
                preds = torch.argmax(logits, dim=1).cpu().numpy()
                all_preds.extend(preds)
                all_targets.extend(y.cpu().numpy())
        val_loss /= len(val_loader.dataset)
        scheduler.step(val_loss)
        
        acc = accuracy_score(all_targets, all_preds)
        prec, rec, f1, _ = precision_recall_fscore_support(all_targets, all_preds, average='macro', zero_division=0)
        print(f'Epoch {epoch+1:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {acc:.4f} | Val F1: {f1:.4f}')
        
        if f1 > best_f1:
            best_f1 = f1
            best_model_state = pickle.dumps(model.state_dict())
            best_val_report = {
                'accuracy': acc,
                'precision': prec,
                'recall': rec,
                'f1_macro': f1,
                'val_loss': val_loss,
                'train_loss': train_loss
            }
            best_cm = confusion_matrix(all_targets, all_preds)
            
    best_model = HybridModel(tabular_dim=len(tabular_cols)).to(device)
    best_model.load_state_dict(pickle.loads(best_model_state))
    return best_model, best_val_report, best_cm, scaler

## 5. Train Hybrid A and Hybrid B Models

In [ ]:
le = LabelEncoder()
y_encoded = le.fit_transform(df_cnn['growth_category'])
indices = np.arange(len(df_cnn))

train_indices, val_indices, y_train_enc, y_val_enc = train_test_split(
    indices, y_encoded, test_size=0.2, stratify=y_encoded, random_state=42
)
df_train = df_cnn.iloc[train_indices].reset_index(drop=True)
df_val = df_cnn.iloc[val_indices].reset_index(drop=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

features_osm_only = [
    'building_count', 'building_density', 'road_density',
    'road_intersection_count', 'green_ratio', 'distance_to_center', 'distance_to_highway'
]
features_osm_spectral = features_osm_only + ['mean_ndvi', 'mean_ndbi', 'mean_ndwi']

print('=== Training Hybrid A (OSM only) ===')
model_a, report_a, cm_a, scaler_a = train_model(
    'Hybrid A', features_osm_only, df_train, df_val, train_indices, val_indices, y_train_enc, y_val_enc, device
)

print('\n=== Training Hybrid B (OSM + Spectral) ===')
model_b, report_b, cm_b, scaler_b = train_model(
    'Hybrid B', features_osm_spectral, df_train, df_val, train_indices, val_indices, y_train_enc, y_val_enc, device
)

## 6. Confusion Matrix (Best Model)

In [ ]:
if report_b['f1_macro'] > report_a['f1_macro']:
    best_model = model_b
    best_cm = cm_b
    best_scaler = scaler_b
    best_cols = features_osm_spectral
    best_name = 'Hybrid B (OSM + Spectral)'
else:
    best_model = model_a
    best_cm = cm_a
    best_scaler = scaler_a
    best_cols = features_osm_only
    best_name = 'Hybrid A (OSM only)'

classes = ['High', 'Low', 'Medium']
plt.figure(figsize=(6, 5))
sns.heatmap(best_cm, annot=True, fmt='d', cmap='Oranges', xticklabels=classes, yticklabels=classes)
plt.title(f'Confusion Matrix: {best_name}')
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

## 7. Grad-CAM Visualization

In [ ]:
best_model.eval()
gradcam = GradCAM(best_model, best_model.backbone.features[-1])
X_val_tab_scaled = best_scaler.transform(df_val[best_cols])

fig, axes = plt.subplots(3, 2, figsize=(10, 15))
categories = ['High', 'Low', 'Medium']

for idx, (cat_name, cat_code) in enumerate(zip(categories, [0, 1, 2])):
    matches = np.where(y_val_enc == cat_code)[0]
    sample_idx = matches[0]
    
    img_rel_path = df_val.iloc[sample_idx]['image_path']
    patch = np.load(project_root / img_rel_path)
    x_img = torch.tensor(patch, dtype=torch.float32).unsqueeze(0).to(device)
    x_tab = torch.tensor(X_val_tab_scaled[sample_idx], dtype=torch.float32).unsqueeze(0).to(device)
    
    cam = gradcam(x_img, x_tab, cat_code)
    
    rgb = patch[[2, 1, 0], :, :]
    rgb = np.transpose(rgb, (1, 2, 0))
    rgb_min = rgb.min(axis=(0, 1), keepdims=True)
    rgb_max = rgb.max(axis=(0, 1), keepdims=True)
    rgb_scaled = (rgb - rgb_min) / (rgb_max - rgb_min + 1e-8)
    rgb_scaled = np.clip(rgb_scaled, 0.0, 1.0)
    
    axes[idx, 0].imshow(rgb_scaled)
    axes[idx, 0].set_title(f'Original RGB - {cat_name} Growth')
    axes[idx, 0].axis('off')
    
    axes[idx, 1].imshow(rgb_scaled)
    axes[idx, 1].imshow(cam, cmap='jet', alpha=0.5)
    axes[idx, 1].set_title(f'Grad-CAM Heatmap - {cat_name} Growth')
    axes[idx, 1].axis('off')

gradcam.release()
plt.tight_layout()
plt.show()

## 8. Final Model Comparison
We compare the baseline classical models and the two hybrid architectures.

In [ ]:
df_compare = pd.read_csv(project_root / 'results/model_comparison_final.csv')
df_compare

## 9. Final Decision & Recommendations

1. **Performance Breakdown:** The classical models perform exceptionally well, with **Baseline XGBoost** obtaining a **0.9574 macro F1 score**. In comparison, Hybrid A yields **0.3839** and Hybrid B yields **0.4570** F1 scores.
2. **Why Classical ML Wins:** The dataset has relatively high-quality engineered features, but is limited in sample size (3,675 training samples). Classical models are much less prone to underfitting than high-parameter deep learning backbones like EfficientNet, which are also trained on non-multispectral imagery.
3. **Decision:** **NO.** The Hybrid Deep Learning models should **not** replace the classical XGBoost model. The Baseline XGBoost model remains the final champion architecture for this deployment.